In [26]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge

In [3]:
df = pd.read_csv("Exam_Score_Prediction.csv")
df.head()

,student_id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,1,17,male,diploma,2.78,92.9,yes,7.4,poor,coaching,low,hard,58.9
1,2,23,other,bca,3.37,64.8,yes,4.6,average,online videos,medium,moderate,54.8
2,3,22,male,b.sc,7.88,76.8,yes,8.5,poor,coaching,high,moderate,90.3
3,4,20,other,diploma,0.67,48.4,yes,5.8,average,online videos,low,moderate,29.7
4,5,20,female,diploma,0.89,71.6,yes,9.8,poor,coaching,low,moderate,43.7


In [4]:
df = df.drop("student_id", axis=1)

In [5]:
df.head()

,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,17,male,diploma,2.78,92.9,yes,7.4,poor,coaching,low,hard,58.9
1,23,other,bca,3.37,64.8,yes,4.6,average,online videos,medium,moderate,54.8
2,22,male,b.sc,7.88,76.8,yes,8.5,poor,coaching,high,moderate,90.3
3,20,other,diploma,0.67,48.4,yes,5.8,average,online videos,low,moderate,29.7
4,20,female,diploma,0.89,71.6,yes,9.8,poor,coaching,low,moderate,43.7


In [6]:
X = df.drop("exam_score", axis=1)
y = df["exam_score"]
print(X.shape)
print(y.shape)

(20000, 11)
(20000,)


In [8]:
X_train, X_temp , y_train, y_temp = train_test_split(X,y, test_size = 0.3 , random_state = 42)
X_val, X_test , y_val , y_test = train_test_split(X_temp , y_temp , test_size = 0.5, random_state = 42)
print(X_train.shape , y_train.shape)
print(X_val.shape , X_val.shape)
print(X_test.shape , y_test.shape)

(14000, 11) (14000,)
(3000, 11) (3000, 11)
(3000, 11) (3000,)


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               20000 non-null  int64  
 1   gender            20000 non-null  object 
 2   course            20000 non-null  object 
 3   study_hours       20000 non-null  float64
 4   class_attendance  20000 non-null  float64
 5   internet_access   20000 non-null  object 
 6   sleep_hours       20000 non-null  float64
 7   sleep_quality     20000 non-null  object 
 8   study_method      20000 non-null  object 
 9   facility_rating   20000 non-null  object 
 10  exam_difficulty   20000 non-null  object 
 11  exam_score        20000 non-null  float64
dtypes: float64(4), int64(1), object(7)
memory usage: 1.8+ MB


In [18]:
num_cols = df.select_dtypes(include = ["float64","int64"]).columns.tolist()
print(num_cols)
num_cols.remove("exam_score")
print(num_cols)

['age', 'study_hours', 'class_attendance', 'sleep_hours', 'exam_score']
['age', 'study_hours', 'class_attendance', 'sleep_hours']


In [15]:
cat_cols = df.select_dtypes(include = ["object"]).columns.tolist()
print(cat_cols)

['gender', 'course', 'internet_access', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']


In [19]:
scaler = StandardScaler()
scaler.fit(X_train[num_cols])

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.transform(X_train[num_cols])
X_val_scaled[num_cols] = scaler.transform(X_val[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

In [21]:
encoder = OneHotEncoder(
    drop = "first",
    sparse_output = False,
    handle_unknown = "ignore"
)
encoder.fit(X_train[cat_cols])
X_train_cat = encoder.transform(X_train[cat_cols])
X_val_cat = encoder.transform(X_val[cat_cols])
X_test_cat = encoder.transform(X_test[cat_cols])

In [22]:
encoded_cols = encoder.get_feature_names_out(cat_cols)
X_train_cat = pd.DataFrame(X_train_cat, columns = encoded_cols, index=X_train.index)
X_val_cat = pd.DataFrame(X_val_cat , columns = encoded_cols, index=X_val.index)
X_test_cat = pd.DataFrame(X_test_cat, columns = encoded_cols, index=X_test.index)


In [23]:
X_train_final = pd.concat([X_train[num_cols], X_train_cat], axis=1)
X_val_final = pd.concat([X_val[num_cols], X_val_cat],axis=1)
X_test_final = pd.concat([X_test[num_cols], X_test_cat], axis=1)

In [24]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state = 42),
    "XGB": XGBRegressor(random_state=42),
    "Decision Tree": DecisionTreeRegressor(random_state=42)
}
results = {}
for name, model in models.items():
    model.fit(X_train_final, y_train)
    y_val_pred = model.predict(X_val_final)
    
    mse = mean_squared_error (y_val, y_val_pred)
    mae = mean_absolute_error(y_val, y_val_pred)
    r2 = r2_score(y_val , y_val_pred)
    
    results[name] = {"MSE":mse ,"MAE":mae , "R2":r2}
    
    
results_df = pd.DataFrame(results).T.sort_values(by="R2",ascending=False)
results_df.head()

,MSE,MAE,R2
Linear Regression,96.210485,7.880347,0.735088
XGB,109.152897,8.394104,0.699451
Random Forest,111.201205,8.469648,0.693811
Decision Tree,230.792069,12.057853,0.364522


In [27]:
ridge = Ridge()

param_grid = {
    "alpha": [0.01, 0.1, 1, 10, 100]  
}
ridge = Ridge()

param_grid = {
    "alpha": [0.01, 0.1, 1, 10, 100]  
}
grid = GridSearchCV(
    estimator=ridge,
    param_grid=param_grid,
    scoring='r2',        # use R² as the metric
    cv=5,                # 5-fold cross-validation
    n_jobs=-1            # use all cores
)


In [28]:
grid.fit(X_train_final, y_train)
print("Best alpha:", grid.best_params_['alpha'])
print("Best CV R² score:", grid.best_score_)


Best alpha: 1
Best CV R² score: 0.7291735124143649


In [29]:
best_ridge = grid.best_estimator_
y_val_pred = best_ridge.predict(X_val_final)

mae = mean_absolute_error(y_val, y_val_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2 = r2_score(y_val, y_val_pred)

print("Tuned Ridge Regression Performance on Validation Set")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)


Tuned Ridge Regression Performance on Validation Set
MAE: 7.880250686903332
RMSE: 9.808516944940987
R²: 0.735097167854422


In [30]:
dt = DecisionTreeRegressor(random_state=42)

param_grid_dt = {
    'max_depth': [None, 3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_dt = GridSearchCV(
    estimator=dt,
    param_grid=param_grid_dt,
    scoring='r2',
    cv=5,
    n_jobs=-1
)


grid_dt.fit(X_train_final, y_train)


print("Decision Tree Best Params:", grid_dt.best_params_)
print("Decision Tree Best CV R²:", grid_dt.best_score_)


best_dt = grid_dt.best_estimator_
y_val_pred = best_dt.predict(X_val_final)

mae = mean_absolute_error(y_val, y_val_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2 = r2_score(y_val, y_val_pred)

print("Decision Tree Validation Performance")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)
print("-"*40)

Decision Tree Best Params: {'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 2}
Decision Tree Best CV R²: 0.6273022810201135
Decision Tree Validation Performance
MAE: 9.115558966163265
RMSE: 11.385296632583358
R²: 0.6430818703412717
----------------------------------------


In [33]:
rf = RandomForestRegressor(random_state=42)

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', None]  # replaced 'auto' with valid options
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

grid_rf.fit(X_train_final, y_train)

print("Random Forest Best Params:", grid_rf.best_params_)
print("Random Forest Best CV R²:", grid_rf.best_score_)

best_rf = grid_rf.best_estimator_
y_val_pred = best_rf.predict(X_val_final)

mae = mean_absolute_error(y_val, y_val_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2 = r2_score(y_val, y_val_pred)

print("Random Forest Validation Performance")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)
print("-"*40)

Random Forest Best Params: {'max_depth': 10, 'max_features': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}
Random Forest Best CV R²: 0.6826352050096387
Random Forest Validation Performance
MAE: 8.429439857938474
RMSE: 10.483300735885894
R²: 0.6973950712852
----------------------------------------


In [32]:
from xgboost import XGBRegressor

xgb = XGBRegressor(random_state=42, verbosity=0)

param_grid_xgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.8, 1],
    'colsample_bytree': [0.7, 0.8, 1]
}

grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid_xgb,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

grid_xgb.fit(X_train_final, y_train)

print("XGBoost Best Params:", grid_xgb.best_params_)
print("XGBoost Best CV R²:", grid_xgb.best_score_)

best_xgb = grid_xgb.best_estimator_
y_val_pred = best_xgb.predict(X_val_final)

mae = mean_absolute_error(y_val, y_val_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2 = r2_score(y_val, y_val_pred)

print("XGBoost Validation Performance")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)
print("-"*40)


XGBoost Best Params: {'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.7}
XGBoost Best CV R²: 0.7238556767381767
XGBoost Validation Performance
MAE: 7.927319828608194
RMSE: 9.883849403989153
R²: 0.7310124697965412
----------------------------------------


In [34]:

final_model = best_ridge  

y_test_pred = final_model.predict(X_test_final)

mae = mean_absolute_error(y_test, y_test_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2 = r2_score(y_test, y_test_pred)

print("Final Model Test Set Performance")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)


Final Model Test Set Performance
MAE: 7.83375068606865
RMSE: 9.706226219555058
R²: 0.7372022334958767
